In [ ]:
import pandas
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification,BertTokenizer, BertForSequenceClassification
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding

dataset_path = r"../../sentiment_dataset.csv"

ModuleNotFoundError: No module named 'transformers'

In [ ]:
dataset = pandas.read_csv(dataset_path)

In [ ]:
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset["label"])

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [ ]:
train_dataset

In [ ]:
checkpoint = "distilbert-base-uncased"
checkpoint = "vinai/phobert-base" 
## https://huggingface.co/vinai/phobert-base 
# Lưu ý: vinai/phobert-base phải max_length = 256 

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
# Ensure consistent label names for training + deploy
model.config.id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
model.config.label2id = {'NEGATIVE': 0, 'POSITIVE': 1}
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# PhoBERT (RoBERTa-style) requires fixed max_length to avoid shape mismatches.
max_length = 256

def tokenizer_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
        return_token_type_ids=False,
    )


def preprocessing(ds):
    ds = ds.map(tokenizer_fn, batched=True, remove_columns=["text"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

In [ ]:
print(train_dataset['text'][0])

In [ ]:
tokenized_train = preprocessing(train_dataset)
tokenized_test = preprocessing(test_dataset)

In [ ]:
import numpy as np
import torch
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds)
        , 'f1': f1_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir='./bert-finetuned-negative-positive',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


In [ ]:
train_result = trainer.train()
metrics = trainer.evaluate()
print(metrics)


In [ ]:
finetuned_model_path = '../../output/bert-finetuned-negative-positive'

In [ ]:
token_save = tokenizer.save_pretrained(finetuned_model_path)
# trained = trainer.save_model("./bert-finetuned-negative-positive")
trained = model.save_pretrained(finetuned_model_path)